# A spare EMA of the student

In distillation the student is trained by the losses, and its weights jump around from
step to step: every SGD update follows one noisy batch. A **spare EMA** is a second
copy of the student whose weights are an exponential moving average of the trained
one, updated after every optimizer step:

```
ema <- m * ema + (1 - m) * student        (m close to 1, e.g. 0.98)
```

The EMA is never trained directly, so it averages out the noise of the last ~1 / (1 - m)
steps. It often generalises better than the raw student, and costs nothing to keep
apart from memory and one extra forward pass. It is the model we export at the end.

Nothing new is needed in `lwfm`: three existing pieces compose into it.

| Piece | Role |
|---|---|
| a frozen `ModelNode` named `student_ema` | holds the averaged weights; gets no gradients, stays in eval mode |
| `EMAUpdate("student", "student_ema")` | moves the EMA towards the student after every optimizer step |
| `ModelCheckpoint(node="student_ema")` + an output sink | saves the EMA's weights and records where they went |

1. The data
2. A teacher to distil from
3. The graph: teacher, student and its spare EMA
4. Losses and metrics
5. Training, with the EMA updated and exported
6. What was saved
7. Loading the exported EMA as a plain `nn.Module`
8. Notes

Run from `examples/`. `lwfm` isn't an installed package yet, so put the repository root on the import path.

In [1]:
import sys
from pathlib import Path

# The notebook runs from examples/, so the repository root (which holds the lwfm
# package) is one level up. Putting it first on sys.path makes `import lwfm` work
# without installing the package.
sys.path.insert(0, str(Path.cwd().parent))

## 1. The data

The same synthetic task as the `Trainer` tutorial: four classes, each a fixed random
"prototype" image, and every sample a noisy copy of its class's prototype. The noise is
heavy, so single batches pull the student in different directions: exactly the noise
an EMA smooths out.

In [2]:
import torch

K = 4  # number of classes
# One fixed random 3x8x8 image per class. A dedicated, seeded generator keeps the
# prototypes identical on every run, independent of the global torch seed.
prototypes = torch.rand(K, 3, 8, 8, generator=torch.Generator().manual_seed(0))


def make_batches(count: int, batch_size: int = 32, seed: int = 0) -> list[dict]:
    """`count` batches of `batch_size` noisy prototype images with their class labels,
    in lwfm's batch layout. The same seed always gives the same batches."""
    generator = torch.Generator().manual_seed(seed)
    batches = []
    for _ in range(count):
        # Random labels in [0, K), shape [batch_size]
        labels = torch.randint(0, K, (batch_size,), generator=generator)
        # Gaussian noise with the image's shape, [batch_size, 3, 8, 8]
        noise = torch.randn(batch_size, 3, 8, 8, generator=generator)
        # A sample is its class's prototype plus noise. prototypes[labels] picks one
        # prototype per sample. The noise (std 1.5) is larger than the prototypes'
        # values (in [0, 1]), so the task is hard enough that the student stays noisy.
        images = prototypes[labels] + 1.5 * noise
        # lwfm's batch layout: graph inputs under "inputs", ground truth under
        # "targets". Nodes, losses and metrics read them as "inputs.image" and
        # "targets.class".
        batches.append({"inputs": {"image": images}, "targets": {"class": labels}})
    return batches


# Any iterable of batch dicts works as a loader; plain lists keep the demo small.
# In a real run these would be torch DataLoaders.
train_loader = make_batches(20)  # 20 batches make one epoch
val_loader = make_batches(10, seed=1)  # new samples from the same classes

## 2. A teacher to distil from

A small MLP trained with plain cross-entropy. In a real run this would be a large
pretrained network; here it only needs to be clearly better than the student.

In [3]:
from torch import nn
from torchmetrics.classification import MulticlassAccuracy

from lwfm.core.graph import ExecutionGraph, ModelNode
from lwfm.core.losses import LossBinding, LossComposer
from lwfm.evaluation.metrics import MetricBinding
from lwfm.training import Trainer, seed_everything


def accuracy(name: str, node: str) -> MetricBinding:
    """Top-1 accuracy of `node`'s logits, reported as `name`."""
    # A MetricBinding pairs a torchmetrics object with the names it reads: here the
    # node's predictions and the ground-truth labels. "micro" averages over samples,
    # i.e. plain accuracy.
    return MetricBinding(
        name,
        MulticlassAccuracy(K, average="micro"),
        [f"{node}.logits", "targets.class"],
    )


# Seed before building the model: weight initialisation draws from the global RNG
seed_everything(0)
# The teacher: a one-hidden-layer MLP on the full 8x8 image. A ModelNode wraps any
# nn.Module and names its input ("inputs.image") and output ("logits"), so the
# output is published to the rest of the graph as "teacher.logits".
teacher_node = ModelNode(
    "teacher",
    nn.Sequential(nn.Flatten(), nn.Linear(3 * 8 * 8, 64), nn.ReLU(), nn.Linear(64, K)),
    inputs="inputs.image",
    outputs="logits",
)
# Plain supervised training: one node, one cross-entropy loss, 5 epochs of 20 steps
teacher_result = Trainer(device="cpu", max_epochs=5).fit(
    ExecutionGraph([teacher_node]),
    train_loader,
    # The loss reads the teacher's logits and the labels by name
    LossComposer(
        [LossBinding("ce", nn.CrossEntropyLoss(), ["teacher.logits", "targets.class"])]
    ),
    torch.optim.SGD(teacher_node.parameters(), lr=0.1),
    # Validate after every epoch, so history holds 5 (step, metrics) entries
    val_loader=val_loader,
    metrics=[accuracy("top1", "teacher")],
)
teacher_result.history[-1]  # (step, metrics) of the final validation

(100, {'top1': 0.890625})

## 3. The graph: teacher, student and its spare EMA

Three nodes read the same image:

```
inputs.image --+--> teacher      (frozen)  --> teacher.logits     --+
               |                                                    +--> loss: ce + kd
               +--> student      (trained) --> student.logits     --+
               |         |
               |         |  EMAUpdate, after every optimizer step
               |         v
               +--> student_ema  (frozen)  --> student_ema.logits ----> metric only
```

- The **student** is a cheaper model than the teacher: it sees the image at half
  resolution.
- The **EMA** starts as an exact copy of the student (`copy.deepcopy`), so the two
  have the same architecture, which `EMAUpdate` checks before training.
- `frozen=True` on the EMA means no gradients and eval mode throughout. It also means
  the EMA has no trainable parameters, so the optimizer is built over the student alone
  and the trainer's "every trainable parameter is in the optimizer" check still passes.

In [4]:
import copy


def make_student_module() -> nn.Module:
    """The student architecture: average-pool to 4x4, then a linear classifier."""
    # A function rather than a single instance, so section 7 can build a fresh,
    # identical architecture to load the exported weights into
    return nn.Sequential(nn.AvgPool2d(2), nn.Flatten(), nn.Linear(3 * 4 * 4, K))


# The teacher for distillation: a deep copy of the trained MLP, so later changes to
# teacher_node can't affect it. frozen=True turns off its gradients and keeps it in
# eval mode, even when the trainer calls graph.train().
teacher = ModelNode(
    "teacher",
    copy.deepcopy(teacher_node.module),  # the trained weights, now frozen
    inputs="inputs.image",
    outputs="logits",
    frozen=True,
)
seed_everything(0)  # a reproducible random initialisation for the student
# The student: the only node the optimizer trains
student = ModelNode(
    "student", make_student_module(), inputs="inputs.image", outputs="logits"
)
# The spare EMA: a deep copy of the student, made before any training, so it starts
# from exactly the student's initial weights. It is frozen: only EMAUpdate changes
# its weights, never the optimizer. Its output is published as "student_ema.logits".
student_ema = ModelNode(
    "student_ema",
    copy.deepcopy(student.module),  # same architecture and starting weights
    inputs="inputs.image",
    outputs="logits",
    frozen=True,
)
# All three nodes read inputs.image, so none depends on another and the graph runs
# them in any valid order. Every node runs on every batch.
graph = ExecutionGraph([teacher, student, student_ema])
list(graph.nodes)

['teacher', 'student', 'student_ema']

## 4. Losses and metrics

The losses read only the teacher and the student: the EMA never contributes a
gradient. It is the standard Hinton loss,
`0.5 * CE(student, labels) + 0.5 * T^2 * KL(softmax(teacher/T) || softmax(student/T))`.

The metrics are where the EMA appears: validation scores the student, its EMA and the
teacher side by side, so we can see whether the EMA is actually better. `top1_ema` is
also what decides which EMA weights get saved.

In [5]:
import torch.nn.functional as F


class KDLoss(nn.Module):
    """Hinton distillation: T^2 * KL(softmax(teacher/T) || softmax(student/T))."""

    def __init__(self, temperature: float = 4.0):
        super().__init__()
        # A temperature T > 1 softens both distributions, so the student also learns
        # how the teacher ranks the wrong classes, not only its top prediction
        self.t = temperature

    def forward(self, student_logits, teacher_logits):
        # Log-probabilities of both models at temperature T, shape [batch, K]
        s = F.log_softmax(student_logits / self.t, dim=-1)
        t = F.log_softmax(teacher_logits / self.t, dim=-1)
        # kl_div(input, target) computes KL(target || input). log_target=True says the
        # target is also in log space; "batchmean" divides the sum by the batch size,
        # which is the mathematically correct KL per sample.
        # T^2 keeps the gradient scale comparable to CE as T changes
        return F.kl_div(s, t, log_target=True, reduction="batchmean") * self.t**2


# The total loss is 0.5 * ce + 0.5 * kd. Each LossBinding names the graph outputs
# its loss reads, in the order of the loss module's forward arguments. Neither reads
# student_ema, so the EMA never receives a gradient from the loss.
loss = LossComposer(
    [
        LossBinding(
            "ce", nn.CrossEntropyLoss(), ["student.logits", "targets.class"], weight=0.5
        ),
        LossBinding("kd", KDLoss(), ["student.logits", "teacher.logits"], weight=0.5),
    ]
)
# Validation scores all three nodes on the same batches. The names ("top1",
# "top1_ema", "top1_teacher") are the keys in result.history and in the logged
# events, and "top1_ema" is what the EMA checkpoint monitors below.
metrics = [
    accuracy("top1", "student"),
    accuracy("top1_ema", "student_ema"),
    accuracy("top1_teacher", "teacher"),
]

## 5. Training, with the EMA updated and exported

Three callbacks do all the work. Order matters only within a hook, and here each
callback acts at a different point:

| Callback | Hook | Effect |
|---|---|---|
| `EMAUpdate("student", "student_ema", momentum=0.98)` | `on_after_optimizer_step` | `ema <- 0.98 * ema + 0.02 * student` |
| `ModelCheckpoint(node="student_ema", monitored_metric="top1_ema", subdir="checkpoints/ema", save_last=False)` | `on_validation_end` | saves the EMA's weights to `checkpoints/ema/best.pt` whenever `top1_ema` improves |
| `ModelCheckpoint(node="student")` | `on_fit_end` | saves the full training state, EMA included, to `checkpoints/last.pt` |

The two checkpoints use different `subdir`s; otherwise both would write
`checkpoints/best.pt` and one would overwrite the other.

**Choosing `m`.** The EMA averages over roughly `1 / (1 - m)` steps: 50 steps for
`m = 0.98`. The run is 160 optimizer steps, so the average covers about a third of it.
With `m = 0.999` (a common value for long ImageNet runs) the EMA would still be mostly
its random starting weights at the end of this short run.

The learning rate is deliberately high (0.5, with momentum), so the raw student is noisy
and the EMA's advantage is easy to see. `fit` runs inside `OutputRouter(...)`, which
passes every event, including the checkpoints' `artifact` events, to
`LocalFileSink`, and records the run's final status.

In [6]:
import tempfile

from lwfm.results.router import OutputRouter
from lwfm.results.sinks import LocalFileSink
from lwfm.training.callbacks import EMAUpdate, ModelCheckpoint

# A fresh run directory under a temporary folder, so re-running the notebook never
# collides with an earlier run. LocalFileSink creates it, and refuses a directory
# that already holds a run.
run_dir = Path(tempfile.mkdtemp()) / "kd_with_ema"

callbacks = [
    # Keep the spare EMA up to date after every optimizer step:
    # student_ema <- 0.98 * student_ema + 0.02 * student
    EMAUpdate("student", "student_ema", momentum=0.98),
    # Export: the EMA's weights alone, whenever its accuracy improves. It writes
    # run_dir/checkpoints/ema/best.pt. save_last=False because the checkpoint below
    # already saves the full state, which includes the EMA.
    ModelCheckpoint(
        run_dir,
        node="student_ema",
        monitored_metric="top1_ema",  # the EMA's own validation accuracy
        subdir="checkpoints/ema",  # kept apart from the student's files
        save_last=False,
    ),
    # For resuming or inspection: the full training state when fit ends. With no
    # monitored_metric, it writes only run_dir/checkpoints/last.pt.
    ModelCheckpoint(run_dir, node="student"),
]

# OutputRouter forwards every event (scalars, and the checkpoints' artifact events)
# to its sinks, here only LocalFileSink. Running fit inside the `with` means the
# run's final status (completed, failed or interrupted) is recorded on exit.
with OutputRouter(LocalFileSink(run_dir)) as sink:
    # 8 epochs x 20 batches = 160 optimizer steps, validated every 20 steps
    result = Trainer(device="cpu", max_epochs=8, val_every_n_steps=20).fit(
        graph,
        train_loader,
        loss,
        # Only the student is optimised; the teacher and the EMA are frozen.
        # A high learning rate with momentum makes the raw student noisy on purpose.
        torch.optim.SGD(student.parameters(), lr=0.5, momentum=0.9),
        val_loader=val_loader,
        metrics=metrics,
        callbacks=callbacks,
        output_sink=sink,
    )

# result.history holds one (step, metrics) pair per validation. Print the three
# models' accuracies side by side to compare the student with its EMA.
print(f"{'step':>4}  {'student':>7}  {'ema':>7}  {'teacher':>7}")
for step, values in result.history:
    print(
        f"{step:>4}  {values['top1']:>7.3f}  {values['top1_ema']:>7.3f}"
        f"  {values['top1_teacher']:>7.3f}"
    )

step  student      ema  teacher
  20    0.575    0.600    0.891
  40    0.544    0.606    0.891
  60    0.522    0.609    0.891
  80    0.556    0.594    0.891
 100    0.550    0.594    0.891
 120    0.528    0.600    0.891
 140    0.519    0.609    0.891
 160    0.541    0.606    0.891


The raw student swings from one validation to the next; the EMA is steadier and higher,
because each of its weights averages the student's last ~50 updates. `result.best`
picks the validation where each model peaked:

In [7]:
# best(metric) returns the validation with the highest value of `metric`, with every
# metric from that validation plus the step it ran at. The EMA's best is what
# checkpoints/ema/best.pt holds.
result.best("top1"), result.best("top1_ema")

({'top1': 0.574999988079071,
  'top1_ema': 0.6000000238418579,
  'top1_teacher': 0.890625,
  'step': 20},
 {'top1': 0.5218750238418579,
  'top1_ema': 0.609375,
  'top1_teacher': 0.890625,
  'step': 60})

## 6. What was saved

The checkpoints write the files first, then send an `artifact` event holding the path
(relative to the run directory, so the directory can be moved). `LocalFileSink` records
those events in `events.jsonl` next to the scalars.

In [8]:
import json

# Every checkpoint file the run wrote, relative to the run directory
print(sorted(str(path.relative_to(run_dir)) for path in run_dir.rglob("*.pt")))
# events.jsonl has one JSON event per line. The artifact events record each save:
# its name (checkpoint/best or checkpoint/last), its path relative to run_dir, and
# the optimizer step it was saved at. best.pt was overwritten at each improvement.
for line in (run_dir / "events.jsonl").open():
    event = json.loads(line)
    if event["kind"] == "artifact":
        print(event["name"], "->", event["value"], "at step", event["step"])

['checkpoints/ema/best.pt', 'checkpoints/last.pt']
checkpoint/best -> checkpoints/ema/best.pt at step 20
checkpoint/best -> checkpoints/ema/best.pt at step 40
checkpoint/best -> checkpoints/ema/best.pt at step 60
checkpoint/last -> checkpoints/last.pt at step 160


`checkpoints/ema/best.pt` holds `deployable_model("student_ema").state_dict()`: the
student architecture's weights and nothing else, no teacher or graph. `last.pt` holds
the whole graph, so the final EMA weights are in there too, under the `student_ema`
node.

In [9]:
# weights_only=True loads tensors and plain containers only, never arbitrary pickled
# objects, so loading a checkpoint can't run code. Both files are saved to allow it.
ema_weights = torch.load(run_dir / "checkpoints/ema/best.pt", weights_only=True)
# Keys are relative to the student module: "2." is the Linear layer at index 2 of
# the nn.Sequential (AvgPool2d and Flatten have no weights)
print("ema/best.pt:", list(ema_weights))

# last.pt is TrainingState.state_dict(): graph, loss, optimizer, scheduler, scaler
# and counters. Its "graph" entry holds every node's weights, prefixed by node name.
last = torch.load(run_dir / "checkpoints/last.pt", weights_only=True)
print(
    "last.pt EMA keys:", [k for k in last["graph"] if k.startswith("nodes.student_ema")]
)

ema/best.pt: ['2.weight', '2.bias']
last.pt EMA keys: ['nodes.student_ema.module.2.weight', 'nodes.student_ema.module.2.bias']


## 7. Loading the exported EMA as a plain `nn.Module`

The file holds weights, not code: build the student architecture, then load them into
it. From there it's an ordinary `nn.Module`, with no trace of the EMA machinery.
Evaluating it reproduces the best `top1_ema` from training.

In [10]:
from lwfm.evaluation.evaluator import Evaluator

# Rebuild the architecture, then load the EMA's weights into it. load_state_dict is
# strict by default, so a key or shape mismatch would raise instead of loading
# silently wrong weights.
exported = make_student_module()
exported.load_state_dict(ema_weights)
exported.eval()  # inference mode for layers like Dropout and BatchNorm

# Wrap it in a one-node graph only to reuse the Evaluator and the metric. The node is
# named "student" here, so the metric reads "student.logits"; the name is only a
# label and has nothing to do with the training graph.
evaluation = Evaluator().evaluate(
    ExecutionGraph(
        [ModelNode("student", exported, inputs="inputs.image", outputs="logits")]
    ),
    val_loader,
    [accuracy("top1", "student")],
    device=torch.device("cpu"),
)
# Should equal result.best("top1_ema")["top1_ema"]: the same weights on the same data
evaluation.metric_values

{'top1': 0.609375}

## 8. Notes

- **Extra compute.** `ExecutionGraph` runs every node on every batch, so the EMA does
  one forward pass (without gradients) per training batch even though no loss reads it.
  That is roughly a third of the cost of a student training step.
- **`m` depends on run length.** The EMA covers about `1 / (1 - m)` optimizer steps.
  Long runs use 0.999 to 0.9999; short runs need smaller values, or the EMA ends up
  dominated by the student's early weights.
- **BatchNorm.** `EMAUpdate` averages parameters but *copies* buffers (BatchNorm running
  statistics) from the student, since averaging running statistics isn't meaningful.
- **Fixed-schedule comparisons.** When every run in a comparison must keep its final
  weights rather than a hand-picked best epoch (e.g. kd_plan's E0 to E3), drop
  `monitored_metric` and keep `save_last=True` on the EMA checkpoint, or read the EMA
  from `last.pt` as in section 6.
- **Same idea, other targets.** `EMAUpdate` is not specific to students: with a teacher
  as the target it gives mean-teacher / DINO-style self-distillation.